In [4]:
import numpy as np
import pandas as pd

WINDOW = 14
HORIZON = 7
STRIDE_FAILED = 1
STRIDE_HEALTHY = 7
STRIDE_EVAL = 2
TARGET_RATIO = 30
SEED = 42
SUBSAMPLE_TRAIN = False
MAX_FILL = 3          # fill gaps of up to 3 missing days

data = pd.read_csv("../data/processed/st12000nm0008_2026q1q2_clean.csv.gz",
                   parse_dates=["date"])
fail_dates = pd.read_csv("../data/processed/fail_dates.csv", parse_dates=["fail_date"])
split_map = pd.read_csv("../data/processed/drive_split.csv")

fail_date_lookup = dict(zip(fail_dates["serial_number"], fail_dates["fail_date"]))
split_lookup = dict(zip(split_map["serial_number"], split_map["split"]))

FEATURE_COLS = [c for c in data.columns if c.startswith("smart_")]
MODEL_COLS = FEATURE_COLS + ["gap_filled"]     # 22 SMART columns + 1 flag = 23

print("rows:", len(data), "| drives:", data["serial_number"].nunique())
print("SMART features:", len(FEATURE_COLS), "| model features:", len(MODEL_COLS))

rows: 3335595 | drives: 18652
SMART features: 22 | model features: 23


In [5]:
def make_windows_for_drive(df_drive, fail_date, window=WINDOW, horizon=HORIZON, stride=1):
    df_drive = df_drive.sort_values("date")
    full = pd.date_range(df_drive["date"].min(), df_drive["date"].max())
    observed = full.isin(df_drive["date"]).astype(np.float32)

    d = df_drive.set_index("date")[FEATURE_COLS].reindex(full).ffill(limit=MAX_FILL)
    feats = np.concatenate([d.values.astype(np.float32),
                            (1 - observed)[:, None]], axis=1)

    valid = ~np.isnan(feats).any(axis=1)
    bad_cum = np.concatenate([[0], np.cumsum(~valid)])
    dates = full.values
    n = len(full)
    windows, labels, end_dates = [], [], []

    for start in range(0, n - window + 1, stride):
        end = start + window - 1
        if bad_cum[end + 1] - bad_cum[start] > 0:
            continue

        label = 0
        if fail_date is not None:
            delta = (fail_date - pd.Timestamp(dates[end])).days
            if delta < 1:
                continue
            if delta <= horizon:
                label = 1

        windows.append(feats[start:end + 1])
        labels.append(label)
        end_dates.append(dates[end].astype("datetime64[D]"))

    return windows, labels, end_dates

In [6]:
rng = np.random.default_rng(SEED)
results = {"train": [], "val": [], "test": []}

for serial, df_drive in data.groupby("serial_number"):
    split = split_lookup.get(serial)
    if split is None:
        continue

    fail_date = fail_date_lookup.get(serial)
    if split == "train":
        stride = STRIDE_FAILED if fail_date is not None else STRIDE_HEALTHY
    else:
        stride = STRIDE_EVAL

    windows, labels, end_dates = make_windows_for_drive(df_drive, fail_date, stride=stride)
    for w, l, e in zip(windows, labels, end_dates):
        results[split].append((w, l, serial, e))

def report(name, items):
    n = len(items)
    pos = sum(t[1] for t in items)
    drives_pos = len({t[2] for t in items if t[1] == 1})
    print(f"{name}: {n} windows | positive: {pos} | ratio 1:{(n - pos) / max(pos, 1):.0f} "
          f"| failed drives with a positive window: {drives_pos}")

print("WINDOWS BUILT:")
for s in ["train", "val", "test"]:
    report(s, results[s])

WINDOWS BUILT:
train: 319722 windows | positive: 873 | ratio 1:365 | failed drives with a positive window: 137
val: 232027 windows | positive: 96 | ratio 1:2416 | failed drives with a positive window: 27
test: 232399 windows | positive: 98 | ratio 1:2370 | failed drives with a positive window: 30


In [7]:
def subsample(items, target_ratio, rng):
    pos = [t for t in items if t[1] == 1]
    neg = [t for t in items if t[1] == 0]
    n_neg = min(len(neg), len(pos) * target_ratio)
    idx = rng.choice(len(neg), size=n_neg, replace=False)
    combined = pos + [neg[i] for i in idx]
    order = rng.permutation(len(combined))
    return [combined[i] for i in order]

if SUBSAMPLE_TRAIN:
    results["train"] = subsample(results["train"], TARGET_RATIO, rng)

print("FINAL (train subsampled:", SUBSAMPLE_TRAIN, "):")
for s in ["train", "val", "test"]:
    report(s, results[s])

FINAL (train subsampled: False ):
train: 319722 windows | positive: 873 | ratio 1:365 | failed drives with a positive window: 137
val: 232027 windows | positive: 96 | ratio 1:2416 | failed drives with a positive window: 27
test: 232399 windows | positive: 98 | ratio 1:2370 | failed drives with a positive window: 30


In [8]:
def stack_split(items):
    X = np.stack([t[0] for t in items])
    y = np.array([t[1] for t in items], dtype=np.float32)
    serial = np.array([t[2] for t in items])
    end_date = np.array([t[3] for t in items], dtype="datetime64[D]")
    return X, y, serial, end_date

X_train, y_train, s_train, d_train = stack_split(results["train"])
X_val,   y_val,   s_val,   d_val   = stack_split(results["val"])
X_test,  y_test,  s_test,  d_test  = stack_split(results["test"])
del results

def prep(X):
    return np.log1p(np.clip(X, 0, None)).astype(np.float32)

n_feat = X_train.shape[-1]
Xtr_log = prep(X_train)
feat_mean = Xtr_log.reshape(-1, n_feat).mean(axis=0)
feat_std = Xtr_log.reshape(-1, n_feat).std(axis=0)
feat_std[feat_std == 0] = 1.0

def normalize(X):
    return ((prep(X) - feat_mean) / feat_std).astype(np.float32)

X_train_n = normalize(X_train)
X_val_n = normalize(X_val)
X_test_n = normalize(X_test)

print("train:", X_train_n.shape, "| val:", X_val_n.shape, "| test:", X_test_n.shape)

train: (319722, 14, 23) | val: (232027, 14, 23) | test: (232399, 14, 23)


In [9]:
assert X_train_n.shape[-1] == len(MODEL_COLS), "feature count does not match MODEL_COLS"

for name, X, y in [("train", X_train_n, y_train), ("val", X_val_n, y_val), ("test", X_test_n, y_test)]:
    print(f"{name}: NaN={int(np.isnan(X).sum())} | inf={int(np.isinf(X).sum())} "
          f"| positives={int(y.sum())} | max|x|={np.abs(X).max():.1f}")

assert not (set(s_train) & set(s_val)), "leak: train/val share drives"
assert not (set(s_train) & set(s_test)), "leak: train/test share drives"
assert not (set(s_val) & set(s_test)), "leak: val/test share drives"
print("no drive appears in two splits: OK")

train: NaN=0 | inf=0 | positives=873 | max|x|=51.6
val: NaN=0 | inf=0 | positives=96 | max|x|=21.0
test: NaN=0 | inf=0 | positives=98 | max|x|=21.0
no drive appears in two splits: OK


In [10]:
out = "../data/processed/"
np.savez_compressed(out + "windows_train.npz", X=X_train_n, y=y_train, serial=s_train, end_date=d_train)
np.savez_compressed(out + "windows_val.npz",   X=X_val_n,   y=y_val,   serial=s_val,   end_date=d_val)
np.savez_compressed(out + "windows_test.npz",  X=X_test_n,  y=y_test,  serial=s_test,  end_date=d_test)

np.savez(out + "normalization_stats.npz", mean=feat_mean, std=feat_std,
         feature_cols=np.array(MODEL_COLS), log1p=np.array(True),
         max_fill=np.array(MAX_FILL))

print("saved.")

saved.


In [11]:
g = data.groupby("serial_number")["date"].agg(["min", "max", "count"])
f = g.loc[g.index.intersection(list(fail_date_lookup))].copy()
f["fail_date"] = f.index.map(fail_date_lookup)
f["split"] = f.index.map(split_lookup)
f["days_before_fail"] = (f["fail_date"] - f["min"]).dt.days
f["span_days"] = (f["max"] - f["min"]).dt.days + 1
f["missing_days"] = f["span_days"] - f["count"]
f["last_row_to_fail"] = (f["fail_date"] - f["max"]).dt.days

print("failed drives:", len(f))
print("days between last row and failure date:")
print(f["last_row_to_fail"].value_counts().head())
print("calendar days of history before failure:")
print(f["days_before_fail"].describe())
print("failed drives with >= 15 days of history:", int((f["days_before_fail"] >= 15).sum()))
print("failed drives with missing days inside their history:", int((f["missing_days"] > 0).sum()))
print(f.groupby("split")["days_before_fail"].apply(lambda s: int((s >= 15).sum())))

failed drives: 274
days between last row and failure date:
last_row_to_fail
0     268
1       2
3       2
2       1
12      1
Name: count, dtype: int64
calendar days of history before failure:
count    274.000000
mean      86.806569
std       52.103474
min        0.000000
25%       40.000000
50%       95.000000
75%      128.750000
max      180.000000
Name: days_before_fail, dtype: float64
failed drives with >= 15 days of history: 246
failed drives with missing days inside their history: 152
split
test      40
train    171
val       35
Name: days_before_fail, dtype: int64


In [12]:
import numpy as np

# 1) Are whole calendar days missing for the entire fleet?
per_day = data.groupby("date").size()
all_days = pd.date_range(data["date"].min(), data["date"].max())
print("calendar days:", len(all_days), "| days with data:", len(per_day))
print("days with no data at all:", [d.date() for d in all_days.difference(per_day.index)])
print("days with unusually few rows:")
print(per_day[per_day < per_day.median() * 0.9])

# 2) Gaps inside each drive's history
dd = data[["serial_number", "date"]].sort_values(["serial_number", "date"]).copy()
dd["gap"] = dd.groupby("serial_number")["date"].diff().dt.days - 1   # missing days before this row
dd["failed"] = dd["serial_number"].isin(set(fail_date_lookup))

gaps = dd[dd["gap"] > 0]
print("\ndrives with at least one gap, by failed (True) / healthy (False):")
print(gaps.groupby("failed")["serial_number"].nunique())
print("\ngap length (missing days) distribution:")
print(gaps["gap"].value_counts().sort_index().head(10))

# 3) Gaps in the last 21 days before failure (what hurts the positive windows)
dd["fail_date"] = dd["serial_number"].map(fail_date_lookup)
dd["days_to_fail"] = (dd["fail_date"] - dd["date"]).dt.days
recent = dd[dd["failed"] & (dd["days_to_fail"] <= 21) & (dd["gap"] > 0)]
print("\nfailed drives with a gap in their last 21 days:", recent["serial_number"].nunique())

calendar days: 181 | days with data: 181
days with no data at all: []
days with unusually few rows:
Series([], dtype: int64)

drives with at least one gap, by failed (True) / healthy (False):
failed
False    4136
True      152
Name: serial_number, dtype: int64

gap length (missing days) distribution:
gap
1.0     3779
2.0     1189
3.0      487
4.0      384
5.0      165
6.0      110
7.0       53
8.0       21
9.0       64
10.0      11
Name: count, dtype: int64

failed drives with a gap in their last 21 days: 130
